# Tema 08 — Laboratorio 07: Normalizar CSV y exportar a JSON

**Objetivo**  
Procesar un CSV de inventario de empresa, normalizar datos, validar puertos, separar registros rechazados y exportar el resultado a JSON.

**Nota de trabajo**  
Ejecuta las celdas en orden. Los ejemplos usan rutas relativas como `Tema08/data`; por tanto, antes de ejecutar los notebooks debe existir esa carpeta con los ficheros de datos indicados en el manual.

## Preparación del laboratorio
En terminal, el flujo general del manual es actualizar el repositorio, entrar en `~/Curso_Python/Tema08`, activar el entorno virtual y abrir Visual Studio Code. En este notebook se ejecutan los bloques Python de forma progresiva.

## Paso previo. Comprobar `inventario_empresa.csv`
Este CSV empresarial será la entrada del proceso de normalización y exportación.

In [ ]:
from pathlib import Path
entrada_csv = Path("Tema08/data/inventario_empresa.csv")
if entrada_csv.exists():
    print(entrada_csv.read_text(encoding="utf-8").strip())
else:
    print(f"No existe {entrada_csv}. Debe copiarse en la carpeta data.")

## Paso 1. Importaciones y configuración inicial

In [ ]:
import csv
import json
from pathlib import Path

carpeta = Path("Tema08/data")
carpeta.mkdir(parents=True, exist_ok=True)
entrada_csv = carpeta / "inventario_empresa.csv"
salida_json = carpeta / "inventario_empresa_normalizado.json"
salida_rechazados = carpeta / "inventario_empresa_rechazados.csv"

print("=== 1. Comprobar fichero de entrada ===")
if not entrada_csv.exists():
    raise FileNotFoundError(f"No existe el fichero de entrada: {entrada_csv}")
print("Entrada:", entrada_csv)

## Paso 2. 2. Funciones auxiliares

In [ ]:
print("\n=== 2. Funciones auxiliares ===")
def normalizar_texto(valor):
    """Limpia espacios y convierte texto a minúsculas."""
    return valor.strip().lower()

def validar_puerto(puerto_txt):
    """Convierte y valida un puerto."""
    puerto = int(puerto_txt)
    if not 1 <= puerto <= 65535:
        raise ValueError("Puerto fuera de rango")
    return puerto

def normalizar_fila(fila):
    """Normaliza una fila del CSV de inventario."""
    return {
        "departamento": normalizar_texto(fila["departamento"]),
        "servidor": normalizar_texto(fila["servidor"]),
        "servicio": normalizar_texto(fila["servicio"]),
        "puerto": validar_puerto(fila["puerto"].strip()),
        "estado": normalizar_texto(fila["estado"]),
        "criticidad": normalizar_texto(fila["criticidad"]),
    }

print("Funciones definidas.")

## Paso 3. 3. Leer, normalizar y validar CSV

In [ ]:
print("\n=== 3. Leer, normalizar y validar CSV ===")
registros_validos = []
registros_rechazados = []
with open(entrada_csv, "r", newline="", encoding="utf-8") as filecsv:
    lector = csv.DictReader(filecsv)
    for numero_linea, fila in enumerate(lector, start=2):
        try:
            normalizada = normalizar_fila(fila)
        except ValueError as error:
            rechazada = dict(fila)
            rechazada["linea"] = numero_linea
            rechazada["motivo"] = str(error)
            registros_rechazados.append(rechazada)
        else:
            registros_validos.append(normalizada)

print("Registros válidos:", len(registros_validos))
print("Registros rechazados:", len(registros_rechazados))

## Paso 4. 4. Construir estructura JSON final

In [ ]:
print("\n=== 4. Construir estructura JSON final ===")
salida = {
    "origen": str(entrada_csv),
    "total_validos": len(registros_validos),
    "total_rechazados": len(registros_rechazados),
    "servicios": registros_validos,
}
print(json.dumps(salida, indent=4, ensure_ascii=False))

## Paso 5. 5. Escribir JSON normalizado

In [ ]:
print("\n=== 5. Escribir JSON normalizado ===")
with open(salida_json, "w", encoding="utf-8") as filejson:
    json.dump(salida, filejson, indent=4, ensure_ascii=False)
print("JSON generado:", salida_json)

## Paso 6. 6. Escribir CSV de rechazados

In [ ]:
print("\n=== 6. Escribir CSV de rechazados ===")
campos_rechazados = ["linea", "departamento", "servidor", "servicio", "puerto", "estado", "criticidad", "motivo"]
with open(salida_rechazados, "w", newline="", encoding="utf-8") as filecsv:
    escritor = csv.DictWriter(filecsv, fieldnames=campos_rechazados)
    escritor.writeheader()
    escritor.writerows(registros_rechazados)
print("CSV de rechazados generado:", salida_rechazados)

## Paso 7. 7. Leer el JSON generado para comprobarlo

In [ ]:
print("\n=== 7. Leer el JSON generado para comprobarlo ===")
with open(salida_json, "r", encoding="utf-8") as filejson:
    comprobacion = json.load(filejson)

for servicio in comprobacion["servicios"]:
    print(f"{servicio['servidor']} -> {servicio['servicio']}:{servicio['puerto']} [{servicio['estado']}]")

**Resultado esperado:** el alumno debe integrar rutas, CSV, JSON, validación, normalización y gestión de registros rechazados en un proceso de datos empresarial.